# PI-GNODE Cross-Region Generalization on Colab

Trains PI-GNODE on high-elevation NDWS events, then on low-elevation, then evaluates each model on both regions.  Output: 4-cell metric matrix + heatmap PNG.

**Runtime:** Google Colab free tier (T4 GPU). Total wall-clock ~1.5–2 hours.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Clone the repo and install dependencies

In [ ]:
# Replace with your team's GitHub URL if forked
REPO_URL = 'https://github.com/josephyu12/pignode-wildfire.git'

!rm -rf /content/pignode-wildfire
!git clone {REPO_URL} /content/pignode-wildfire
%cd /content/pignode-wildfire

In [ ]:
# Install the project + PyG. Colab already has torch + CUDA so we just
# match versions for the PyG extensions.
!pip install -q -e .
import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH_VER}  cuda={CUDA_VER}  gpu={torch.cuda.is_available()}')
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html

## 2. Download NDWS data

From HuggingFace (~2.4 GB total).  Takes ~5–10 min.

In [ ]:
import os, subprocess
os.makedirs('data/raw/ndws', exist_ok=True)

# HuggingFace mirror of Huot et al. NDWS, zarr-zipped.
from huggingface_hub import snapshot_download
snap = snapshot_download(
    repo_id='TheRootOf3/next-day-wildfire-spread',
    repo_type='dataset',
    local_dir='data/raw/ndws/_hf',
    allow_patterns=['*.zarr.zip', '*.zip'],
)
print('downloaded to', snap)

# Unzip each split's zarr archive into the layout the loader expects
import zipfile, glob
for split in ('train', 'eval', 'test'):
    matches = glob.glob(f'{snap}/**/{split}*.zip', recursive=True) + \
              glob.glob(f'{snap}/{split}*.zip')
    if not matches:
        raise RuntimeError(f'no zip found for split={split}; check {snap}')
    z = matches[0]
    target = f'data/raw/ndws/{split}.zarr'
    if os.path.isdir(target):
        continue
    print(f'unzipping {z} -> {target}')
    with zipfile.ZipFile(z) as zf:
        zf.extractall(target)

!ls -la data/raw/ndws/

In [ ]:
# Compute normalization stats once (cached).
from wildfire.data.ndws import compute_norm_stats, compute_region_assignments
stats = compute_norm_stats()
print('norm stats computed -> data/processed/norm_stats.npz')

# Pre-compute region assignments so the loader doesn't recompute per process.
regions = compute_region_assignments()
print('region assignments:')
print('  threshold elevation:', float(regions['threshold']))
for k in regions:
    if k != 'threshold':
        print(f'  {k}: {len(regions[k])} events')

## 3. Train PI-GNODE on each region

Two trainings of 8 epochs each.  ~30–45 min each on T4.

In [ ]:
# High-elevation training
!python -m wildfire.train --exp pignode_high_elev \
    --model pignode --uniform-edges --hidden 128 --ode-layers 2 \
    --n-eval-steps 1 --t-end 1.0 --epochs 8 --batch-size 16 --lr 3e-4 \
    --eval-batches 30 --region high_elev --device cuda

In [ ]:
# Low-elevation training
!python -m wildfire.train --exp pignode_low_elev \
    --model pignode --uniform-edges --hidden 128 --ode-layers 2 \
    --n-eval-steps 1 --t-end 1.0 --epochs 8 --batch-size 16 --lr 3e-4 \
    --eval-batches 30 --region low_elev --device cuda

## 4. Cross-evaluate the 2x2 matrix

In [ ]:
for trained in ('high_elev', 'low_elev'):
    for evald in ('high_elev', 'low_elev'):
        ckpt = f'experiments/pignode_{trained}/best.pt'
        print(f'\n=== eval: trained on {trained}, tested on {evald} ===')
        !python -m wildfire.eval_region --ckpt {ckpt} --region {evald} --split test --device cuda

## 5. Generate the heatmap figure

In [ ]:
!python -m wildfire.figures
from IPython.display import Image
Image('experiments/_figures/region_split.png')

## 6. Download results back to your computer

Zips just the new region-split outputs (not the raw data) and downloads.

In [ ]:
!zip -r region_split_results.zip \
    experiments/pignode_high_elev/metrics.json \
    experiments/pignode_high_elev/best.pt \
    experiments/pignode_high_elev/eval_test_high_elev.json \
    experiments/pignode_high_elev/eval_test_low_elev.json \
    experiments/pignode_low_elev/metrics.json \
    experiments/pignode_low_elev/best.pt \
    experiments/pignode_low_elev/eval_test_high_elev.json \
    experiments/pignode_low_elev/eval_test_low_elev.json \
    experiments/_figures/region_split.png

from google.colab import files
files.download('region_split_results.zip')